In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc gem-suite matplotlib
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# Nishimori-Phasenübergang

import TutorialFeedback from '@site/src/components/TutorialFeedback';



*Geschätzter Aufwand: 3 Minuten auf einem Heron-r2-Prozessor (HINWEIS: Dies ist nur eine Schätzung. Deine Laufzeit kann abweichen.)*
## Lernziele
Nach dem Durcharbeiten dieses Tutorials sollten Benutzer folgende Ergebnisse erwarten:
- Den Nishimori-Phasenübergang verstehen und wie er sich als das Auftreten langreichweitiger Verschränkung im Random-Bond-Ising-Modell zeigt.
- Das Protokoll zur *Erzeugung von Verschränkung durch Messung* (GEM) auf Quantenhardware mit Mid-Circuit-Messungen und Schaltkreisen konstanter Tiefe implementieren.
- Den Übergang durch Extraktion der Zwei-Punkt-Korrelation und der normierten Varianz der Magnetisierung aus experimentellen Daten charakterisieren.

## Voraussetzungen
Wir empfehlen, die folgenden Themen vor diesem Tutorial zu kennen:
- Den Leitfaden [Qubits messen](/guides/measure-qubits), insbesondere den Abschnitt über Mid-Circuit-Messungen, auf die das GEM-Protokoll angewiesen ist.
- [Exakte und verrauschte Simulation mit Qiskit-Aer-Primitiven](/guides/simulate-with-qiskit-aer), wie der Abschnitt im kleinen Maßstab ausgeführt wird.
- [Langreichweitige Verschränkung mit dynamischen Schaltkreisen](/tutorials/long-range-entanglement), ein begleitendes Tutorial, das dasselbe messungsbasierte Verschränkungsparadigma verwendet.
- [Heavy-Hex-Gitter](https://www.ibm.com/quantum/blog/heavy-hex-lattice), die IBM&reg;-Hardware-Topologie, auf der das Plaquette-Gitter aufgebaut ist.
## Hintergrund
Dieses Tutorial zeigt, wie man einen Nishimori-Phasenübergang auf einem Quantenprozessor realisiert. Das Experiment wurde ursprünglich in [*Realizing the Nishimori transition across the error threshold for constant-depth quantum circuits*](https://arxiv.org/abs/2309.02863) beschrieben.

Der Nishimori-Phasenübergang bezeichnet den Übergang zwischen kurzreichweitig und langreichweitig geordneten Phasen im Random-Bond-Ising-Modell. Auf einem Quantencomputer manifestiert sich die langreichweitig geordnete Phase als ein Zustand, in dem Qubits über das gesamte Gerät verschränkt sind. Dieser hochverschränkte Zustand wird mit dem Protokoll zur *Erzeugung von Verschränkung durch Messung* (GEM) präpariert. Durch den Einsatz von Mid-Circuit-Messungen kann das GEM-Protokoll Qubits über das gesamte Gerät hinweg mit Schaltkreisen konstanter Tiefe verschränken. Dieses Tutorial verwendet die Implementierung des GEM-Protokolls aus dem Softwarepaket [GEM Suite](https://github.com/qiskit-community/gem-suite).
## Anforderungen
Stelle vor Beginn dieses Tutorials sicher, dass Folgendes installiert ist:

- Qiskit SDK v1.0 oder neuer, mit Unterstützung für [Visualisierung](https://docs.quantum.ibm.com/api/qiskit/visualization)
- Qiskit Runtime v0.22 oder neuer (`pip install qiskit-ibm-runtime`)
- Qiskit Aer v0.14 oder neuer (`pip install qiskit-aer`)
- GEM Suite (`pip install gem-suite`)
## Einrichtung

In [1]:
import matplotlib.pyplot as plt
import warnings

from collections import defaultdict

from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_aer import AerSimulator

from qiskit.transpiler import generate_preset_pass_manager

from gem_suite import PlaquetteLattice
from gem_suite.experiments import GemExperiment

## Kleines Simulator-Beispiel

In diesem Abschnitt wird der vollständige Ablauf auf dem rauschfreien `AerSimulator` durchgegangen. Das Plaquette-Gitter wird auf eine einzige Plaquette (12 Qubits) beschränkt, damit die Simulation klein und schnell bleibt und trotzdem jeden Teil des GEM-Protokolls abdeckt: Mid-Circuit-Messung, den $R_{ZZ}$-Winkeldurchlauf, Dekodierung und die normierte-Varianz-Analyse. Derselbe Ablauf wird später auf mehrere Plaquettes und das vollständige Gitter auf echter Hardware skaliert.

### Schritt 1: Klassische Eingaben auf ein Quantenproblem abbilden

Das GEM-Protokoll arbeitet auf einem Quantenprozessor, dessen Qubit-Konnektivität durch ein Gitter beschrieben wird. Aktuelle IBM Quantum&reg;-Prozessoren verwenden das [Heavy-Hex-Gitter](https://www.ibm.com/quantum/blog/heavy-hex-lattice). Die Qubits des Prozessors werden anhand der Einheitszelle des Gitters, die sie besetzen, in *Plaquettes* gruppiert. Da ein Qubit in mehr als einer Einheitszelle vorkommen kann, sind die Plaquettes nicht disjunkt. Auf dem Heavy-Hex-Gitter enthält eine Plaquette 12 Qubits. Die Plaquettes bilden ihrerseits auch ein Gitter, wobei zwei Plaquettes verbunden sind, wenn sie Qubits teilen. Auf dem Heavy-Hex-Gitter teilen benachbarte Plaquettes jeweils 3 Qubits.

Im Softwarepaket GEM Suite ist die grundlegende Klasse zur Implementierung des GEM-Protokolls `PlaquetteLattice`, die das Gitter der Plaquettes repräsentiert (das sich vom Heavy-Hex-Gitter unterscheidet). Eine `PlaquetteLattice` kann aus einer Qubit-Kopplungsmap initialisiert werden. Derzeit werden nur Heavy-Hex-Kopplungsmaps unterstützt.

Die folgende Code-Zelle initialisiert ein Plaquette-Gitter aus der Kopplungsmap einer Quantenverarbeitungseinheit (QPU). Das Plaquette-Gitter umfasst nicht immer die gesamte Hardware. Beispielsweise hat `ibm_torino` insgesamt 133 Qubits, aber das größte Plaquette-Gitter, das auf das Gerät passt, verwendet nur 125 davon und umfasst 18 Plaquettes; `ibm_pittsburgh` (156 Qubits) passt ähnlich 144 Qubits in 21 Plaquettes. Dasselbe Muster gilt für andere Heavy-Hex-QPUs mit unterschiedlichen Qubit-Zahlen.

In [ ]:
# QiskitRuntimeService.save_account(channel="ibm_quantum", token="<YOUR_API_KEY>", overwrite=True,
# set_as_default=True)
service = QiskitRuntimeService()
backend = service.least_busy(
    operational=True, simulator=False, min_num_qubits=127
)
aer_backend = AerSimulator.from_backend(backend)
plaquette_lattice = PlaquetteLattice.from_coupling_map(backend.coupling_map)

print(f"Number of qubits in backend: {backend.num_qubits}")
print(
    f"Number of qubits in plaquette lattice: {len(list(plaquette_lattice.qubits()))}"
)
print(f"Number of plaquettes: {len(list(plaquette_lattice.plaquettes()))}")

Du kannst das Plaquette-Gitter visualisieren, indem du ein Diagramm seiner Graphdarstellung erzeugst. Im Diagramm werden die Plaquettes durch beschriftete Sechsecke dargestellt, und zwei Plaquettes sind durch eine Kante verbunden, wenn sie Qubits teilen.

In [3]:
plaquette_lattice.draw_plaquettes()

<Image src="../docs/images/tutorials/nishimori-phase-transition/extracted-outputs/625882a4-faeb-4d96-b441-c989f43c4dea-0.avif" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/tutorials/nishimori-phase-transition/extracted-outputs/625882a4-faeb-4d96-b441-c989f43c4dea-0.avif)

Mit der Methode `plaquettes` kannst du Informationen über einzelne Plaquettes abrufen, z. B. welche Qubits sie enthalten.

In [4]:
# Get a list of the plaquettes
plaquettes = list(plaquette_lattice.plaquettes())
# Display information about plaquette 0
plaquettes[0]

PyPlaquette(index=0, qubits=[3, 4, 5, 6, 7, 16, 17, 23, 24, 25, 26, 27], neighbors=[4, 3, 1])

You can also produce a diagram of the underlying qubits that form the plaquette lattice.

In [5]:
plaquette_lattice.draw_qubits()

<Image src="../docs/images/tutorials/nishimori-phase-transition/extracted-outputs/a19d63ce-3572-4081-a008-c1332fbbe303-0.avif" alt="Output of the previous code cell" />

Du kannst auch ein Diagramm der zugrunde liegenden Qubits erzeugen, die das Plaquette-Gitter bilden.

In [6]:
# Filter the plaquette lattice down to a single plaquette (12 qubits)
# so the AerSimulator run stays fast. The full lattice is used later
# in the large-scale hardware example.
gem_exp = GemExperiment(plaquette_lattice.filter([9]), backend=aer_backend)

# visualize the plaquette lattice after filtering
plaquette_lattice.filter([9]).draw_qubits()

<Image src="../docs/images/tutorials/nishimori-phase-transition/extracted-outputs/02357c6e-5c83-4ac0-811d-22602d9f33d5-0.avif" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/tutorials/nishimori-phase-transition/extracted-outputs/a19d63ce-3572-4081-a008-c1332fbbe303-0.avif)

Neben den Qubit-Bezeichnungen und den Kanten, die anzeigen, welche Qubits verbunden sind, enthält das Diagramm drei weitere Informationen, die für das GEM-Protokoll relevant sind:
- Jedes Qubit ist entweder schattiert (grau) oder nicht schattiert. Die schattierten Qubits sind „Site"-Qubits, die die Gitterplätze des Ising-Modells repräsentieren, und die nicht schattierten Qubits sind „Bond"-Qubits, die zur Vermittlung von Wechselwirkungen zwischen den Site-Qubits dienen.
- Jedes Site-Qubit ist entweder mit (A) oder (B) beschriftet und gibt damit eine von zwei Rollen an, die ein Site-Qubit im GEM-Protokoll übernehmen kann (die Rollen werden später erläutert).
- Jede Kante ist in einer von sechs Farben eingefärbt, wodurch die Kanten in sechs Gruppen aufgeteilt werden. Diese Aufteilung bestimmt, wie Zwei-Qubit-Gates parallelisiert werden können, sowie verschiedene Scheduling-Muster, die auf einem verrauschten Quantenprozessor wahrscheinlich unterschiedlich viel Fehler verursachen. Da Kanten innerhalb einer Gruppe disjunkt sind, kann eine Schicht von Zwei-Qubit-Gates gleichzeitig auf diesen Kanten angewendet werden. Tatsächlich lassen sich die sechs Farben in drei Gruppen zu je zwei Farben aufteilen, sodass die Vereinigung jeder Gruppe von zwei Farben immer noch disjunkt ist. Daher werden nur drei Schichten von Zwei-Qubit-Gates benötigt, um jede Kante zu aktivieren. Es gibt 12 Möglichkeiten, die sechs Farben so aufzuteilen, und jede solche Aufteilung ergibt ein anderes 3-Schicht-Gate-Schedule.

Nachdem du ein Plaquette-Gitter erstellt hast, besteht der nächste Schritt darin, ein `GemExperiment`-Objekt zu initialisieren, dem sowohl das Plaquette-Gitter als auch das Backend übergeben werden, auf dem du das Experiment ausführen möchtest. Die Klasse `GemExperiment` verwaltet die eigentliche Implementierung des GEM-Protokolls, einschließlich der Erzeugung von Schaltkreisen, des Einreichens von Jobs und der Analyse der Daten. Die folgende Code-Zelle initialisiert die Experiment-Klasse und schränkt das Plaquette-Gitter dabei auf eine einzige Plaquette (12 Qubits) ein, damit die Simulation klein und schnell bleibt. Das vollständige Plaquette-Gitter wird später beim Skalieren auf echte Hardware verwendet.

In [7]:
circuits = gem_exp.circuits()
print(f"Total number of circuits: {len(circuits)}")

Total number of circuits: 252


![Output of the previous code cell](../docs/images/tutorials/nishimori-phase-transition/extracted-outputs/02357c6e-5c83-4ac0-811d-22602d9f33d5-0.avif)

Ein GEM-Protokoll-Schaltkreis wird mithilfe der folgenden Schritte aufgebaut:
1. Bereite den Zustand $|+\rangle$ für alle Qubits vor, indem du auf jedes Qubit ein Hadamard-Gate anwendest.
2. Wende ein $R_{ZZ}$-Gate zwischen jedem Paar verbundener Qubits an. Dies kann mit drei Gate-Schichten erreicht werden. Jedes $R_{ZZ}$-Gate wirkt auf ein Site-Qubit und ein Bond-Qubit. Wenn das Site-Qubit mit (B) beschriftet ist, ist der Winkel auf $\frac{\pi}{2}$ festgelegt. Wenn das Site-Qubit mit (A) beschriftet ist, kann der Winkel variieren und so verschiedene Schaltkreise erzeugen. Standardmäßig ist der Winkelbereich auf 21 gleichmäßig verteilte Punkte zwischen $0$ und $\frac{\pi}{2}$ (einschließlich) eingestellt.
3. Miss jedes Bond-Qubit in der Pauli-$X$-Basis. Da Qubits in der Pauli-$Z$-Basis gemessen werden, kann dies erreicht werden, indem vor der Messung des Qubits ein Hadamard-Gate angewendet wird.

Beachte, dass das in der Einleitung dieses Tutorials zitierte Paper eine andere Konvention für den $R_{ZZ}$-Winkel verwendet, die sich von der in diesem Tutorial verwendeten Konvention um einen Faktor 2 unterscheidet.

In Schritt 3 werden nur die Bond-Qubits gemessen. Um zu verstehen, in welchem Zustand die Site-Qubits verbleiben, ist es hilfreich, den Fall zu betrachten, dass der in Schritt 2 auf Site-Qubits (A) angewendete $R_{ZZ}$-Winkel gleich $\frac{\pi}{2}$ ist. In diesem Fall befinden sich die Site-Qubits in einem hochverschränkten Zustand ähnlich dem GHZ-Zustand,

$$
\lvert \text{GHZ} \rangle = \lvert 00 \cdots 00 \rangle + \lvert 11 \cdots 11 \rangle.
$$

Aufgrund der Zufälligkeit der Messergebnisse kann der tatsächliche Zustand der Site-Qubits ein anderer Zustand mit langreichweitiger Ordnung sein, beispielsweise $\lvert 00110 \rangle + \lvert 11001 \rangle$. Der GHZ-Zustand kann jedoch durch eine Dekodierungsoperation auf Basis der Messergebnisse wiederhergestellt werden. Wenn der $R_{ZZ}$-Winkel von $\frac{\pi}{2}$ heruntergeregelt wird, kann die langreichweitige Ordnung noch bis zu einem kritischen Winkel wiederhergestellt werden, der ohne Rauschen bei ungefähr $0{,}3 \pi$ liegt. Unterhalb dieses Winkels weist der resultierende Zustand keine langreichweitige Verschränkung mehr auf. Dieser Übergang zwischen dem Vorhandensein und dem Fehlen langreichweitiger Ordnung ist der Nishimori-Phasenübergang.

In der obigen Beschreibung wurden die Site-Qubits ungemessen gelassen, und die Dekodierungsoperation kann durch Anwenden von Quantengattern durchgeführt werden. Im Experiment, wie es in der GEM Suite implementiert ist, werden die Site-Qubits tatsächlich gemessen, und die Dekodierungsoperation wird in einem klassischen Nachverarbeitungsschritt angewendet.

In der obigen Beschreibung kann die Dekodierungsoperation durch Anwenden von Quantengattern auf die Site-Qubits durchgeführt werden, um den Quantenzustand wiederherzustellen. Wenn das Ziel jedoch darin besteht, den Zustand sofort zu messen – beispielsweise zu Charakterisierungszwecken –, dann kannst du die Site-Qubits zusammen mit den Bond-Qubits messen und die Dekodierungsoperation in einem klassischen Nachverarbeitungsschritt anwenden.

Zusätzlich zur Abhängigkeit vom $R_{ZZ}$-Winkel in Schritt 2, der standardmäßig über 21 Werte durchläuft, hängt der GEM-Protokoll-Schaltkreis auch vom verwendeten Scheduling-Muster ab, mit dem die drei Schichten von $R_{ZZ}$-Gates implementiert werden. Wie bereits erwähnt, gibt es 12 solcher Scheduling-Muster. Daher beträgt die Gesamtanzahl der Schaltkreise im Experiment $21 \times 12 = 252$.

Die Schaltkreise des Experiments können mit der Methode `circuits` der Klasse `GemExperiment` erzeugt werden.

In [8]:
# Restrict experiment to the first scheduling pattern
gem_exp.set_experiment_options(schedule_idx=0)

# There are less circuits now
circuits = gem_exp.circuits()
print(f"Total number of circuits: {len(circuits)}")

# Print the RZZ angles swept over
print(f"RZZ angles:\n{gem_exp.parameters()}")

Total number of circuits: 21
RZZ angles:
[0.         0.07853982 0.15707963 0.23561945 0.31415927 0.39269908
 0.4712389  0.54977871 0.62831853 0.70685835 0.78539816 0.86393798
 0.9424778  1.02101761 1.09955743 1.17809725 1.25663706 1.33517688
 1.41371669 1.49225651 1.57079633]


The following code cell draws a diagram of the circuit at index 5. To reduce the size of the diagram, the measurement gates at the end of the circuit are removed.

In [9]:
# Get the circuit at index 5
circuit = circuits[5]
# Remove the final measurements to ease visualization
circuit.remove_final_measurements()
# Draw the circuit
circuit.draw("mpl", fold=-1, scale=0.5)

<Image src="../docs/images/tutorials/nishimori-phase-transition/extracted-outputs/fd57d483-c70b-4ad5-b309-15750ad38bac-0.avif" alt="Output of the previous code cell" />

Für die Zwecke dieses Tutorials reicht es aus, nur ein einziges Scheduling-Muster zu betrachten. Die folgende Code-Zelle schränkt das Experiment auf das erste Scheduling-Muster ein. Dadurch hat das Experiment nur noch 21 Schaltkreise, einen für jeden durchlaufenen $R_{ZZ}$-Winkel.

In [10]:
# Demonstrate setting transpile options
gem_exp.set_transpile_options(
    optimization_level=1  # This is the default optimization level
)
pass_manager = generate_preset_pass_manager(
    backend=aer_backend,
    initial_layout=list(gem_exp.physical_qubits),
    **dict(gem_exp.transpile_options),
)
transpiled = pass_manager.run(circuit)
transpiled.draw("mpl", idle_wires=False, fold=-1, scale=0.5)

<Image src="../docs/images/tutorials/nishimori-phase-transition/extracted-outputs/e9b99d48-8d33-46b5-bff5-480ab1c1c1f2-0.avif" alt="Output of the previous code cell" />

### Step 3: Execute using Qiskit primitives

To execute the GEM protocol circuits on the hardware, call the `run` method of the `GemExperiment` object. You can specify the number of shots you want to sample from each circuit. The `run` method returns an [ExperimentData](https://qiskit-community.github.io/qiskit-experiments/stubs/qiskit_experiments.framework.ExperimentData.html) object which you should save to a variable. Note that the `run` method only submits jobs without waiting for them to finish, so it is a non-blocking call.

In [11]:
exp_data = gem_exp.run(shots=10_000)

Die folgende Code-Zelle zeichnet ein Diagramm des Schaltkreises mit Index 5. Um die Größe des Diagramms zu reduzieren, werden die Messgatter am Ende des Schaltkreises entfernt.

In [12]:
# The noiseless AerSimulator produces zero-variance UFloat objects in the
# analysis, which triggers a harmless warning from the `uncertainties`
# library. Suppress it so the output stays clean.
with warnings.catch_warnings():
    warnings.filterwarnings(
        "ignore", message="Using UFloat objects with std_dev==0"
    )
    exp_data.block_for_results()
exp_data

ExperimentData(GemExperiment, 90bf2a90-f729-4c4e-a6da-664aecb11039, job_ids=['04a7c405-47fd-46ca-aa4b-aaf7e339cfbe'], metadata=<5 items>, figure_names=['two_point_correlation.svg', 'normalized_variance.svg', 'plaquette_ops.svg', 'bond_ops.svg'])

![Output of the previous code cell](../docs/images/tutorials/nishimori-phase-transition/extracted-outputs/fd57d483-c70b-4ad5-b309-15750ad38bac-0.avif)

### Schritt 2: Problem für die Ausführung auf Quantenhardware optimieren
Das Transpilieren von Quantenschaltkreisen für die Ausführung auf Hardware umfasst typischerweise eine [Reihe von Phasen](/guides/transpiler-stages). Die Phasen, die den größten Rechenaufwand verursachen, sind in der Regel die Auswahl des Qubit-Layouts, das Routing der Zwei-Qubit-Gates gemäß der Qubit-Konnektivität der Hardware sowie die Optimierung des Schaltkreises zur Minimierung der Gate-Anzahl und -Tiefe. Im GEM-Protokoll sind die Layout- und Routing-Phasen nicht erforderlich, da die Hardware-Konnektivität bereits in den Entwurf des Protokolls eingeflossen ist. Die Schaltkreise haben bereits ein Qubit-Layout, und die Zwei-Qubit-Gates sind bereits auf native Verbindungen abgebildet. Darüber hinaus sollte, um die Struktur des Schaltkreises bei Variation des $R_{ZZ}$-Winkels zu erhalten, nur eine sehr grundlegende Schaltkreisoptimierung durchgeführt werden.

Die Klasse `GemExperiment` transpiliert Schaltkreise bei der Ausführung des Experiments transparent. Die Layout- und Routing-Phasen sind standardmäßig bereits so überschrieben, dass sie nichts tun, und die Schaltkreisoptimierung wird auf einem Niveau durchgeführt, das nur Einzelqubit-Gates optimiert. Du kannst jedoch zusätzliche Optionen mit der Methode `set_transpile_options` überschreiben oder übergeben. Zur Veranschaulichung transpiliert die folgende Code-Zelle den zuvor angezeigten Schaltkreis manuell und zeichnet den transpilierten Schaltkreis.

In [13]:
def magnetization_distribution(
    counts_dict: dict[str, int],
) -> dict[str, float]:
    """Compute magnetization distribution from counts dictionary."""
    # Construct dictionary from magnetization to count
    mag_dist = defaultdict(float)
    for bitstring, count in counts_dict.items():
        mag = bitstring.count("0") - bitstring.count("1")
        mag_dist[mag] += count
    # Normalize
    shots = sum(counts_dict.values())
    for mag in mag_dist:
        mag_dist[mag] /= shots
    return mag_dist


# Get counts dictionaries with and without decoding
data = exp_data.data()
# Get the last data point, which is at the angle for the GHZ state
raw_counts = data[-1]["counts"]
# Without decoding
site_indices = [
    i for i, q in enumerate(gem_exp.plaquettes.qubits()) if q.role == "Site"
]
site_raw_counts = defaultdict(int)
for key, val in raw_counts.items():
    site_str = "".join(key[-1 - i] for i in site_indices)
    site_raw_counts[site_str] += val
# With decoding
_, site_decoded_counts = gem_exp.plaquettes.decode_outcomes(
    raw_counts, return_counts=True
)

# Compute magnetization distribution
raw_magnetization = magnetization_distribution(site_raw_counts)
decoded_magnetization = magnetization_distribution(site_decoded_counts)

# Plot
plt.bar(*zip(*raw_magnetization.items()), label="raw")
plt.bar(*zip(*decoded_magnetization.items()), label="decoded", width=0.3)
plt.legend()
plt.xlabel("Magnetization")
plt.ylabel("Frequency")
plt.title("Magnetization distribution with and without decoding")

Text(0.5, 1.0, 'Magnetization distribution with and without decoding')

<Image src="../docs/images/tutorials/nishimori-phase-transition/extracted-outputs/8ead3582-16df-4616-836c-bdce867ad6b8-1.avif" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/tutorials/nishimori-phase-transition/extracted-outputs/e9b99d48-8d33-46b5-bff5-480ab1c1c1f2-0.avif)

### Schritt 3: Ausführung mit Qiskit-Primitiven
Um die GEM-Protokoll-Schaltkreise auf der Hardware auszuführen, rufe die Methode `run` des `GemExperiment`-Objekts auf. Du kannst die Anzahl der Shots angeben, die du aus jedem Schaltkreis sampeln möchtest. Die Methode `run` gibt ein [ExperimentData](https://qiskit-community.github.io/qiskit-experiments/stubs/qiskit_experiments.framework.ExperimentData.html)-Objekt zurück, das du in einer Variablen speichern solltest. Beachte, dass die Methode `run` Jobs nur einreicht, ohne auf deren Abschluss zu warten – es handelt sich also um einen nicht-blockierenden Aufruf.

In [14]:
exp_data.figure("two_point_correlation")

<Image src="../docs/images/tutorials/nishimori-phase-transition/extracted-outputs/4ecb25c8-e572-49af-a879-9943039db131-0.avif" alt="Output of the previous code cell" />

Um auf die Ergebnisse zu warten, rufe die Methode `block_for_results` des `ExperimentData`-Objekts auf. Dieser Aufruf lässt den Interpreter warten, bis die Jobs abgeschlossen sind.

In [15]:
exp_data.figure("normalized_variance")

<Image src="../docs/images/tutorials/nishimori-phase-transition/extracted-outputs/2b351d68-3924-445a-94ef-047b16214e8a-0.avif" alt="Output of the previous code cell" />

## Large-scale hardware example

Having validated the protocol on a simulator, you can now scale up the experiment and run it on the real quantum hardware backend selected in the [Setup](#setup) section. This example uses two larger problem sizes:

- **Six plaquettes (~49 qubits)**: a mid-size run that already shows the rightward shift of the critical point under hardware noise.
- **The full plaquette lattice**: every plaquette the device's heavy-hex topology supports (for example, 18 plaquettes / 125 qubits on `ibm_torino` or 21 plaquettes / 144 qubits on `ibm_pittsburgh`), entangling qubits across the entire device with constant-depth circuits.

The single code cell below is self-contained: it builds the plaquette lattice from the backend's coupling map and runs both experiments, so this section can be executed after the [Setup](#setup) cells without first running the small-scale section.

In [ ]:
# -------------------------Step 1-------------------------
# Initialize the runtime service, pick a real quantum hardware backend,
# and build the plaquette lattice from its coupling map. This is repeated
# from the small-scale example so this cell can run standalone after the
# Setup section. The full plaquette lattice is the "large-scale" target;
# a six-plaquette subset (range(3, 9)) is also used to show an intermediate
# scaling step.
service = QiskitRuntimeService()
backend = service.least_busy(
    operational=True, simulator=False, min_num_qubits=127
)
plaquette_lattice = PlaquetteLattice.from_coupling_map(backend.coupling_map)

# Build a GemExperiment for the full plaquette lattice and one for the
# six-plaquette subset, each restricted to a single scheduling pattern so
# the experiment has one circuit per RZZ angle (21 circuits total).
gem_exp_full = GemExperiment(plaquette_lattice, backend=backend)
gem_exp_full.set_experiment_options(schedule_idx=0)
gem_exp_6 = GemExperiment(
    plaquette_lattice.filter(range(3, 9)), backend=backend
)
gem_exp_6.set_experiment_options(schedule_idx=0)

circuits = gem_exp_full.circuits()
print(f"Total number of circuits (full lattice): {len(circuits)}")

# -------------------------Step 2-------------------------
# GemExperiment transpiles internally for the target backend: the layout
# and routing stages are overridden because the plaquette lattice already
# matches the hardware connectivity, and optimization is restricted so the
# RZZ angle structure is preserved. The code below manually transpiles one
# circuit from the six-plaquette experiment with the same settings this
# experiment will use, and draws it for inspection. (The full-lattice
# transpiled circuit has too many qubits to visualize cleanly, so the
# six-plaquette circuit is used here as a representative example.)
gem_exp_6.set_transpile_options(optimization_level=1)
circuits_6 = gem_exp_6.circuits()
pass_manager = generate_preset_pass_manager(
    backend=backend,
    initial_layout=list(gem_exp_6.physical_qubits),
    **dict(gem_exp_6.transpile_options),
)
transpiled = pass_manager.run(circuits_6[5])
display(transpiled.draw("mpl", idle_wires=False, fold=-1, scale=0.5))

# -------------------------Step 3-------------------------
# Run both problem sizes on real hardware:
#   1. Six plaquettes (~49 qubits) — an intermediate scale-up.
#   2. The full plaquette lattice — every plaquette the device supports.
exp_data_6 = gem_exp_6.run(shots=10_000, job_tags=["TUT_NPT"])
exp_data_full = gem_exp_full.run(shots=10_000, job_tags=["TUT_NPT"])
exp_data_6.block_for_results()
exp_data_full.block_for_results()

# -------------------------Step 4-------------------------
# Plot the normalized variance at each scale. The peak marks the critical
# point of the Nishimori transition; as the system grows, hardware noise
# shifts the peak rightward.
display(exp_data_6.figure("normalized_variance"))
exp_data_full.figure("normalized_variance")

Total number of circuits (full lattice): 21


<Image src="../docs/images/tutorials/nishimori-phase-transition/extracted-outputs/08581c09-a6a5-4a56-9fc4-abf22b063c6a-1.avif" alt="Output of the previous code cell" />

<Image src="../docs/images/tutorials/nishimori-phase-transition/extracted-outputs/08581c09-a6a5-4a56-9fc4-abf22b063c6a-2.avif" alt="Output of the previous code cell" />

<Image src="../docs/images/tutorials/nishimori-phase-transition/extracted-outputs/08581c09-a6a5-4a56-9fc4-abf22b063c6a-3.avif" alt="Output of the previous code cell" />

### Schritt 4: Nachverarbeitung und Ausgabe im gewünschten klassischen Format
Bei einem $R_{ZZ}$-Winkel von $\frac{\pi}{2}$ wäre der dekodierte Zustand ohne Rauschen der GHZ-Zustand. Die langreichweitige Ordnung des GHZ-Zustands kann durch Darstellung der Magnetisierung der gemessenen Bitstrings visualisiert werden. Die Magnetisierung $M$ ist definiert als die Summe der Einzel-Qubit-Pauli-$Z$-Operatoren,
$$
M = \sum_{j=1}^N Z_j,
$$
wobei $N$ die Anzahl der Site-Qubits ist. Ihr Wert für einen Bitstring ist gleich der Differenz zwischen der Anzahl der Nullen und der Anzahl der Einsen. Die Messung des GHZ-Zustands liefert mit gleicher Wahrscheinlichkeit den Zustand aus lauter Nullen oder den Zustand aus lauter Einsen, sodass die Magnetisierung halb der Zeit $+N$ und die andere Hälfte der Zeit $-N$ wäre. Bei Fehlern durch Rauschen würden zwar auch andere Werte auftreten, aber wenn das Rauschen nicht zu groß ist, wäre die Verteilung immer noch nahe bei $+N$ und $-N$ konzentriert.

Für die rohen Bitstrings vor der Dekodierung wäre die Verteilung der Magnetisierung äquivalent zu der gleichmäßig zufälliger Bitstrings, ohne Rauschen.

Die folgende Code-Zelle stellt die Magnetisierung der rohen Bitstrings und der dekodierten Bitstrings beim $R_{ZZ}$-Winkel von $\frac{\pi}{2}$ dar.